# Stage 3: the words

**The question.** Does what a video says and shows in writing predict its lift, beyond stage 2's best baseline?
`lift` is stage 1's target: the video's log share rate minus the median of its creator's last 30 videos posted at
least 14 days earlier. Stage 2's best baseline is **B3 ridge**, metadata in ridge regression with no hour, which beat
zero lift for seen creators by 0.0007 log10, all of it from the video's length. This stage starts from B3 and asks
what the words add.

The text fields, all fixed when the video is posted (rule 4):

| Field | What it is | Written by |
|---|---|---|
| Caption | the creator's caption, hashtags included | the creator |
| Transcript | the spoken words, in English | Whisper, translated where the video isn't in English |
| Hook, body, payoff | the script split in three | a language model, from the transcript (or, when nobody speaks, from the on-screen text and scene) |
| On-screen text | the text read off sampled frames | a vision-language model (trap 5) |
| Scene description | what happens in the picture | the same vision-language model (trap 5) |

The scene description is a feature, not a fact: it is a model's reading of the frames, so it may carry the picture
more than the words. The notebook keeps it apart from the rest ("the words") for that reason.

Two kinds of text features, each with the same two heads (ridge on lift, logistic regression on breakout):

- **TF-IDF**: words and word pairs, weighted up when rare across videos and down when common, fitted on the
  training rows only. Sparse, linear, and every weight can be read.
- **Sentence embeddings**: `BAAI/bge-small-en-v1.5`, a small English encoder (33M parameters, 384 numbers per text)
  that runs on this laptop's CPU and that my content platform already uses.

The code is in `src/sharerate/text.py` (the vocabulary, TF-IDF and the embedding cache), with its tests in
`tests/test_text.py`. The loader's `load_videos(text=True)` returns the fields.

## 0. The bar, written before the first run (rule 7)

This cell was written, and committed, before any text model was fitted or scored.

- **The rows** are stage 2's: the 21,332 validation rows of `targets`, from 203 creators. Every text model predicts
  for every row: an empty field is a row of zeros, with a flag saying the field is empty, so no row is dropped and
  rule 5 holds without re-scoring on fewer rows.
- **The reference** is B3 ridge with no hour, refitted exactly as in notebook 02 (cell 24): MAE 0.3681 for seen
  creators. Zero lift (B0) and stage 2's best ranker (B1, the weighted mean with a half-life of 20) are shown beside
  it for context.
- **"On top of the baseline."** The ridge head is fitted on the training rows to what B3 leaves (lift minus B3's
  prediction), and predicts B3 plus its own correction. The logistic head is fitted to breakout with B3's
  predicted lift as one of its inputs. Either way, what the text adds is measured against B3 itself.
- **The primary bar.** A text model (its ridge head) beats the baseline if the 95% paired creator-bootstrap interval of its
  validation MAE gain over B3 ridge (B3's MAE minus its own) lies above 0 for **seen** creators: 138 creators,
  13,646 videos. New creators (27, 2,275 videos) and held-out creators (38, 5,411) are reported and are not
  required to pass.
- **The secondary bar.** For the logistic head, the gain in breakout ROC-AUC over B3 ridge (its predicted lift
  used as the breakout score) has an interval above 0, and so does the gain in the top-10% share.
- **Reposts.** 73 seen-creator validation videos have an exact twin among the training videos (`has_twin`). A pass
  must hold with them removed too; a gain that needs them is memorised reposts, and is reported as a miss.
- **Embeddings against TF-IDF.** bge-small beats TF-IDF if the interval of its MAE gain over the all-fields TF-IDF
  ridge (both on top of B3) lies above 0 for seen creators. Only then does stage 4's rental run a larger text
  encoder; otherwise it runs no text job.
- **Fields one at a time,** and the words apart from the scene description, are descriptive: reported with
  intervals, to say where the signal is, not as extra chances to pass.
- **What goes to stage 5.** The passing model with the largest MAE gain for seen creators. If none passes, stage 5's
  combined model leaves text out, and this notebook says so.
- **Fixed in advance, never tuned:** the vocabulary (words and word pairs that at least 20 training creators use,
  with @mentions and links removed, so no feature is one creator's signature and every weight can be shown under
  rule 9); TF-IDF's settings (1 + log of the count, rows scaled to length 1, IDF from the training rows); the
  encoder (bge-small-en-v1.5, the first 256 tokens, unit-length vectors).
- **Chosen on validation,** over all 21,332 rows: the ridge penalty by MAE, the logistic penalty by ROC-AUC. The
  test split is never opened here; stage 5 opens it once.
- **A miss is a miss.** A gain whose interval touches 0 is reported as "the words add nothing beyond B3", never
  tuned until it passes. That result is publishable.
- **Intervals** come from 2,000 replicates that resample creators, with seed 2026, the same draws as stage 2.